In [ ]:
# ⚡ Avazu CTR Model — Full Dataset (~40M rows)
import pandas as pd
import joblib
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import gc

# Load full dataset (may take 3–5 minutes, ~4GB RAM)
df = pd.read_csv("../data/avazu-ctr.csv")  # no nrows limit

# Drop unhelpful or unique IDs
df = df.drop(columns=['id', 'device_id', 'device_ip'])

# Label
y = df['click']
X = df.drop(columns=['click'])

# Convert all to categorical
for col in X.columns:
    X[col] = X[col].astype('category')

# Garbage collect to free RAM
gc.collect()

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train LightGBM model
model = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=64)
model.fit(X_train, y_train)

# Evaluate
preds = model.predict(X_test)
print(classification_report(y_test, preds))

# Save model
joblib.dump(model, "../backend/models/ctr_model.pkl")
print("✅ Full CTR model saved to models/ctr_model.pkl")
